### Retrieval Metrics

In [5]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import joblib
import os

from sentence_transformers import SentenceTransformer

from sklearn.datasets import fetch_20newsgroups

In [6]:
# load the dataset
newsgroups_train = fetch_20newsgroups(subset='train', shuffle=True, random_state=42, data_home='./dataset')



In [7]:
# convert dataset to dataframe
df = pd.DataFrame({
    'text': newsgroups_train.data,
    'category': newsgroups_train.target
})

In [8]:
df.head()

,text,category
0,From: lerxst@wam.umd.edu (where's my thing)\nS...,7
1,From: guykuo@carson.u.washington.edu (Guy Kuo)...,4
2,From: twillis@ec.ecn.purdue.edu (Thomas E Will...,4
3,From: jgreen@amber (Joe Green)\nSubject: Re: W...,1
4,From: jcm@head-cfa.harvard.edu (Jonathan McDow...,14


In [9]:
df.shape

(11314, 2)

In [10]:
print("\nNumber of Categories:", len(newsgroups_train.target_names))
print("\nCategories:", newsgroups_train.target_names)


Number of Categories: 20

Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


## Preprocessing and vectorizing data

In [11]:
model_name = "BAAI/bge-base-en-v1.5"
model = SentenceTransformer(model_name)

embedding_vectors = joblib.load('embeddings.joblib')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
len(embedding_vectors) # --> 11314

11314

### Basic functions for retrieval

In [13]:
# removing whitte space

def preprocess_text(text):
    text = text.strip()
    return text


def cosine_similarity(v1, array_of_vectors):
    # Handle torch tensors for v1
    if hasattr(v1, "detach"): 
        v1 = v1.detach().cpu().numpy()
    v1 = np.asarray(v1, dtype=np.float32).ravel()

    # Handle torch tensors for array_of_vectors
    if hasattr(array_of_vectors, "detach"):  
        array_of_vectors = array_of_vectors.detach().cpu().numpy()
    A = np.asarray(array_of_vectors, dtype=np.float32)

    if A.ndim == 1:
        A = A.ravel()
        denom = np.linalg.norm(v1) * np.linalg.norm(A)
        return float(0.0 if denom == 0 else np.dot(v1, A) / denom)
    
    # 2D case: compute similarities for each row in A
    A = np.atleast_2d(A)
    v1_norm = np.linalg.norm(v1)
    A_norms = np.linalg.norm(A, axis=1)
    denom = v1_norm * A_norms
    with np.errstate(divide='ignore', invalid='ignore'):
        sims = (A @ v1) / np.where(denom == 0, 1.0, denom)
    sims[denom == 0] = 0.0
    return sims.tolist()





def top_k_greatest_indices(lst, k):
    indexed_list = list(enumerate(lst))
    
    sorted_by_value = sorted(indexed_list, key=lambda x: x[1], reverse=True)
    
    top_k_indices = [index for index, value in sorted_by_value[:k]]
    return top_k_indices

In [14]:
def retrieve_documents(query, embeddings, model, top_k=5):
    
    query_clean = preprocess_text(query)
    query_embedding = model.encode(query_clean, convert_to_tensor=False).astype(np.float32)

    cosine_scores = []
    for x in embeddings:
        
        if hasattr(x, "detach"):  
            x = x.detach().cpu().numpy()
        x = np.asarray(x, dtype=np.float32)

        score = cosine_similarity(query_embedding, x) 
        cosine_scores.append(float(score))

    top_results = top_k_greatest_indices(cosine_scores, k=top_k)

    print(f"Query: {query}")
    for idx in top_results:
        print(f"Document: {df.iloc[idx]['text'][:200]}...")
        print(f"Category: {newsgroups_train.target_names[df.iloc[idx]['category']]}...")
        print("\n\n")


In [15]:
example_query = "space exploration"
retrieve_documents(example_query, embedding_vectors, model, top_k = 2)

Query: space exploration
Document: From: u1452@penelope.sdsc.edu (Jeff Bytof - SIO)
Subject: End of the Space Age?
Organization: San Diego Supercomputer Center @ UCSD
Lines: 16
Distribution: world
NNTP-Posting-Host: penelope.sdsc.edu

...
Category: sci.space...



Document: From: dennisn@ecs.comm.mot.com (Dennis Newkirk)
Subject: Space class for teachers near Chicago
Organization: Motorola
Distribution: usa
Nntp-Posting-Host: 145.1.146.43
Lines: 59

I am posting this for...
Category: sci.space...





### Precision@k

In [16]:
def precision_at_k(relevant_count, k):
    if relevant_count <0 or k<0:
        raise ValueError("All input values must be non negative!")
    
    if k==0:
        return 0.0
    return relevant_count / k 

### Recall@k

In [17]:
def recall_at_k(relevant_count, total_relevant):
    if relevant_count < 0 or total_relevant < 0:
        raise ValueError("All input values must be non negative!")
    if  total_relevant == 0:
        return 0.0
    return relevant_count / total_relevant

## Computing metrics over some queries

In [18]:
test_queries = [
    {"query": "advancements in space exploration technology", "desired_category": "sci.space"},
    {"query": "real-time rendering techniques in computer graphics", "desired_category": "comp.graphics"},
    {"query": "latest findings in cardiovascular medical research", "desired_category": "sci.med"},
    {"query": "NHL playoffs and team performance statistics", "desired_category": "rec.sport.hockey"},
    {"query": "impacts of cryptography in online security", "desired_category": "sci.crypt"},
    {"query": "the role of electronics in modern computing devices", "desired_category": "sci.electronics"},
    {"query": "motorcycles maintenance tips for enthusiasts", "desired_category": "rec.motorcycles"},
    {"query": "high-performance baseball tactics for championships", "desired_category": "rec.sport.baseball"},
    {"query": "historical influence of politics on society", "desired_category": "talk.politics.misc"},
    {"query": "latest technology trends in the Windows operating system", "desired_category": "comp.os.ms-windows.misc"}
    
]


In [19]:
def compute_metrics(queries, embeddings, model, top_k=5):


    results = []

    # Normalize all embeddings to NumPy once
    np_embeddings = []
    for x in embeddings:
        if hasattr(x, "detach"): 
            x = x.detach().cpu().numpy()
        np_embeddings.append(np.asarray(x, dtype=np.float32).ravel())
    E = np.vstack(np_embeddings)  # shape: (N, D)

    for item in queries:
        query = item["query"]
        desired_category = item["desired_category"]

        # Get NumPy, not torch, to avoid GPU->NumPy conversion errors
        q_clean = preprocess_text(query)
        q_emb = model.encode(q_clean, convert_to_tensor=False)
        q_emb = np.asarray(q_emb, dtype=np.float32).ravel()

        # Compute similarities vectorized
        cosine_scores = cosine_similarity(q_emb, E)  # list of floats length N

        # Top-K indices
        top_results = top_k_greatest_indices(cosine_scores, k=top_k)

        # Retrieved categories
        retrieved_categories = [
            newsgroups_train.target_names[df.iloc[idx]["category"]] for idx in top_results
        ]

        # Metrics
        relevant_in_top_k = sum(1 for cat in retrieved_categories if cat == desired_category)
        total_relevant_in_corpus = sum(
            1 for idx in range(len(df))
            if newsgroups_train.target_names[df.iloc[idx]["category"]] == desired_category
        )

        p = precision_at_k(relevant_in_top_k, top_k)
        r = recall_at_k(relevant_in_top_k, total_relevant_in_corpus)

        results.append({
            "query": query,
            "precision@k": p,
            "recall@k": r,
        })

    return results

In [20]:
# Run the queries and compute metrics with different K values
k_values = [5, 20, 50]

for k in k_values:
    print(f"\n{'='*80}")
    print(f"Results with K={k}:")
    print('='*80)
    results = compute_metrics(test_queries, embedding_vectors, model, top_k=k)
    
    # Display the results
    for result in results:
        print(f"Query: {result['query']}")
        print(f"  Precision@{k}: {result['precision@k']:.2f}, Recall@{k}: {result['recall@k']:.2f}")
        print()


Results with K=5:
Query: advancements in space exploration technology
  Precision@5: 1.00, Recall@5: 0.01

Query: real-time rendering techniques in computer graphics
  Precision@5: 1.00, Recall@5: 0.01

Query: latest findings in cardiovascular medical research
  Precision@5: 1.00, Recall@5: 0.01

Query: NHL playoffs and team performance statistics
  Precision@5: 1.00, Recall@5: 0.01

Query: impacts of cryptography in online security
  Precision@5: 1.00, Recall@5: 0.01

Query: the role of electronics in modern computing devices
  Precision@5: 1.00, Recall@5: 0.01

Query: motorcycles maintenance tips for enthusiasts
  Precision@5: 1.00, Recall@5: 0.01

Query: high-performance baseball tactics for championships
  Precision@5: 1.00, Recall@5: 0.01

Query: historical influence of politics on society
  Precision@5: 0.40, Recall@5: 0.00

Query: latest technology trends in the Windows operating system
  Precision@5: 0.80, Recall@5: 0.01


Results with K=20:
Query: advancements in space explor

<b>Understanding the Results:</b>

The results above clearly demonstrate the precision-recall tradeoff in retrieval systems as we vary K from 5 to 20 to 50:

<b>Precision@K Trends (generally decreases as K increases):</b>

- At K=5: Most queries achieve very high precision (0.80-1.00), with 8 out of 10 queries having perfect precision (1.00). This means nearly all retrieved documents are highly relevant.

- At K=20: Precision starts to decline for some queries:

    - "electronics in computing devices" drops to 0.80 (from 1.00)
    - "Windows operating system" drops to 0.65 (from 0.80)
    - "motorcycles maintenance" drops to 0.95 (from 1.00)


- At K=50: Precision decreases further as we retrieve more documents:

    - "computer graphics" drops to 0.88 (from 1.00)
    - "electronics in computing devices" drops to 0.66 (from 0.80)
    - "Windows operating system" drops to 0.60 (from 0.65)
    - "historical influence of politics" remains around 0.50-0.52 (the lowest across all K values)


<b>Recall@K Trends (increases as K increases):</b>

- At K=5: Recall is very low (~0.01 or 1%), meaning only about 1% of all relevant documents are retrieved

- At K=20: Recall triples to ~0.03 (3%), capturing more relevant documents

- At K=50: Recall increases to 0.05-0.08 (5-8%), capturing approximately 8 times more relevant documents than K=5

<b>Key Observations:</b>

The tradeoff is clear: As K increases, we retrieve more of the total relevant documents (higher recall), but at the cost of including some irrelevant documents (lower precision).

Some queries are harder than others: The query "historical influence of politics on society" consistently shows the lowest precision (0.40-0.52), suggesting that this query is semantically ambiguous or the category "talk.politics.misc" is harder to distinguish from related categories.

For RAG systems, K=5 to K=20 is often optimal: These values provide high precision (most retrieved documents are relevant) while keeping the context size manageable for the LLM. Even though recall is low, the goal is to find the most relevant documents, not all relevant documents.

Recall remains relatively low even at K=50: This is expected since each category contains hundreds of documents (500-600), so retrieving 50 documents only captures ~8-10% of the total relevant documents. To achieve high recall, we would need K values in the hundreds, which would severely impact precision and be impractical for RAG applications.